In [1]:
# ============================================
# IMPORT LIBRARIES
# ============================================

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10

2026-05-10 13:58:03.064381: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778421483.300516      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778421483.362071      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778421483.885340      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778421483.885379      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778421483.885382      57 computation_placer.cc:177] computation placer alr

In [2]:
# ============================================
# LOAD CIFAR-10 DATASET (TESTING)
# ============================================

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print("CIFAR Test Shape:", x_test.shape)

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
CIFAR Test Shape: (10000, 32, 32, 3)


In [3]:
# ============================================
# NORMALIZE CIFAR TEST DATA
# ============================================

x_test = x_test.astype("float32") / 255.0

y_test = y_test.flatten()

In [7]:
# ============================================
# LOAD CINIC-10 TRAIN DATASET
# ============================================

cinic_train = tf.keras.utils.image_dataset_from_directory(
    '/kaggle/input/datasets/mengcius/cinic10/train',
    image_size=(32,32),
    batch_size=64,
    shuffle=True
)

Found 90000 files belonging to 10 classes.


I0000 00:00:1778421870.063740      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778421870.069806      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [8]:
# ============================================
# NORMALIZE CINIC TRAIN DATA
# ============================================

cinic_train = cinic_train.map(
    lambda x, y: (
        tf.cast(x, tf.float32) / 255.0,
        tf.cast(y, tf.int32)
    )
)

In [9]:
# ============================================
# DATA AUGMENTATION
# ============================================

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

In [10]:
# ============================================
# INCEPTION MODULE
# ============================================

def inception_module(x, filters):

    branch1 = layers.Conv2D(
        filters,
        (1,1),
        padding='same',
        activation='relu'
    )(x)

    branch2 = layers.Conv2D(
        filters,
        (1,1),
        padding='same',
        activation='relu'
    )(x)

    branch2 = layers.Conv2D(
        filters,
        (3,3),
        padding='same',
        activation='relu'
    )(branch2)

    branch3 = layers.Conv2D(
        filters,
        (1,1),
        padding='same',
        activation='relu'
    )(x)

    branch3 = layers.Conv2D(
        filters,
        (5,5),
        padding='same',
        activation='relu'
    )(branch3)

    branch4 = layers.MaxPooling2D(
        (3,3),
        strides=1,
        padding='same'
    )(x)

    branch4 = layers.Conv2D(
        filters,
        (1,1),
        padding='same',
        activation='relu'
    )(branch4)

    output = layers.Concatenate()([
        branch1,
        branch2,
        branch3,
        branch4
    ])

    return output

In [11]:
# ============================================
# BUILD INCEPTION MODEL
# ============================================

def build_inception(input_shape=(32,32,3), num_classes=10):

    inputs = layers.Input(shape=input_shape)

    x = data_augmentation(inputs)

    x = layers.Conv2D(
        64,
        (3,3),
        padding='same',
        activation='relu'
    )(x)

    x = layers.MaxPooling2D((2,2))(x)

    x = inception_module(x, 32)

    x = inception_module(x, 64)

    x = layers.MaxPooling2D((2,2))(x)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(128, activation='relu')(x)

    x = layers.Dropout(0.3)(x)

    outputs = layers.Dense(
        num_classes,
        activation='softmax'
    )(x)

    model = models.Model(inputs, outputs)

    return model

In [12]:
# ============================================
# CREATE MODEL
# ============================================

model = build_inception()

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 32, 32, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 32, 32, 3) │          0 │ input_layer[0][0] │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 32, 32,    │      1,792 │ sequential[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 16, 16,    │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 16, 16,    │      2,080 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 16, 16,    │      2,080 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 16, 16,    │          0 │ max_pooling2d[0]… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 16, 16,    │      2,080 │ max_pooling2d[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 16, 16,    │      9,248 │ conv2d_2[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 16, 16,    │     25,632 │ conv2d_4[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 16, 16,    │      2,080 │ max_pooling2d_1[… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 16, 16,    │          0 │ conv2d_1[0][0],   │
│ (Concatenate)       │ 128)              │            │ conv2d_3[0][0],   │
│                     │                   │            │ conv2d_5[0][0],   │
│                     │                   │            │ conv2d_6[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 16, 16,    │      8,256 │ concatenate[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_10 (Conv2D)  │ (None, 16, 16,    │      8,256 │ concatenate[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 16, 16,    │          0 │ concatenate[0][0] │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 16, 16,    │      8,256 │ concatenate[0][0] │
│                     │ 64)               │            │                 

 Total params: 251,594 (982.79 KB)

 Trainable params: 251,594 (982.79 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
# ============================================
# COMPILE MODEL
# ============================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [14]:
# ============================================
# TRAIN ON CINIC-10
# ============================================

history = model.fit(
    cinic_train,
    epochs=40,
    validation_data=(x_test, y_test)
)

Epoch 1/40


I0000 00:00:1778421878.426052     152 cuda_dnn.cc:529] Loaded cuDNN version 91002


1407/1407 ━━━━━━━━━━━━━━━━━━━━ 92s 60ms/step - accuracy: 0.2309 - loss: 2.0210 - val_accuracy: 0.4202 - val_loss: 1.5746
Epoch 2/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 26s 18ms/step - accuracy: 0.3729 - loss: 1.6853 - val_accuracy: 0.4970 - val_loss: 1.3708
Epoch 3/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.4218 - loss: 1.5593 - val_accuracy: 0.5168 - val_loss: 1.3214
Epoch 4/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.4501 - loss: 1.4829 - val_accuracy: 0.5302 - val_loss: 1.2916
Epoch 5/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 26s 18ms/step - accuracy: 0.4713 - loss: 1.4330 - val_accuracy: 0.5636 - val_loss: 1.2011
Epoch 6/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 26s 18ms/step - accuracy: 0.4910 - loss: 1.3880 - val_accuracy: 0.5689 - val_loss: 1.1905
Epoch 7/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 26s 18ms/step - accuracy: 0.5059 - loss: 1.3528 - val_accuracy: 0.5760 - val_loss: 1.1892
Epoch 8/40
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 26s 18ms/step - accuracy: 0.5170 - loss: 1.31

In [15]:
# ============================================
# TEST ON CIFAR-10
# ============================================

test_loss, test_acc = model.evaluate(x_test, y_test)

print("CINIC → CIFAR Inception Accuracy:", test_acc)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7219 - loss: 0.8178
CINIC → CIFAR Inception Accuracy: 0.7210999727249146


In [16]:
# ============================================
# SAVE MODEL
# ============================================

model.save('/kaggle/working/cinic_to_cifar_inception.keras')